In [1]:
import pandas as pd
import re
import pdfplumber

# ---------- READ PDF AND EXTRACT TEXT ----------
pdf_file_path = r"C:\Users\Pratik Mali\Downloads\JU_PL_Mst.pdf"


def read_pdf_text(pdf_path: str) -> str:
    """Return concatenated text from all pages of the given PDF."""
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"
    print(f"Reading PDF from: {pdf_path}")
    print("Extracted text from PDF:")
    print("=" * 50)
    print(full_text)
    print("=" * 50)
    return full_text

# The extracted PDF text that will be consumed by the next cell
RAW_TEXT = read_pdf_text(pdf_file_path)


Reading PDF from: C:\Users\Pratik Mali\Downloads\JU_PL_Mst.pdf
Extracted text from PDF:
Detail Sales Order ju
Order 25/ JSO/ JUA/ 4367 Lk Dt: PL No: Cust Req Dt: 17/10/25 Order Dt09/09/25 RF:7.2.2:4
Customer Sterling Jewelers( Outlet) PL Qty: Rev Dt1: 01/01/80 PO No. Ombre Project SRD (1t) 09/09/25
Rec
Design Gross Kt Col GldAs Gold Dia Dia Cs Cs Lab Sales Ord Value Prd Del Dt Exp Del Dt Prd Seq.
sr
Odsr Size/Model Wt Wt Rate Qty Wt No. Wt Val Price Qty
1 ABP08249C 1.52 AG925 AG 0.40 3594.55 3 0.20 15.25 83.18 550 45749.00 01/01/80 17/10/25 SMP
PDT Cust.Inst 1/5CT Fashion Pendant , 1/5ct & above lab created inscribed on the Dia Girdle
1 Plt Rate 1385.00
ABP08249C
Silver Rate 40.74 Prd Inst. All White
SS
Pld Rate 1133.00 Stmp Inst 925 & JD LOGO
Bill of Materails Sz Inst
Sepcial Rem SKU# AG925 FASHION Pendant
Prt Cd. 20 Past Exp Qty: 1
Spl Rem: Part Desc. PENDANT- 1
Sub Rem.
Lst Exp Bill: GrsWt: NetWt: OdKt: Sub PO:
Data Entry Pre Fin Loss Pst Fin Loss Val Addn Ctg Minimum Diamond Wt Max

In [3]:
# ✅ JU parser with FIXED quantity extraction
import pandas as pd
import re

def process_ju_data_with_item_po():
    extracted_text = RAW_TEXT

    priority = input("Enter Priority value (e.g., REG, HIGH, LOW): ").strip() or "REG"

    data = []
    lines = extracted_text.splitlines()
    sr_no = 1
    current_item_po_no = ""

    for i, raw_line in enumerate(lines):
        line = raw_line.strip()
        if not line:
            continue

        # Capture the current ItemPoNo from the nearest preceding Order header
        order_match = re.search(r'Order\s+\d+/\s+\w+/\s+\w+/\s+(\d+)', line)
        if order_match:
            current_item_po_no = order_match.group(1)
            continue

        # Detect item row like: "1 ABP08249C 1.52 AG925 ..."
        item_match = re.match(r'^(\d+)\s+([A-Z0-9]+)\s+([\d.]+)\s+AG925', line)
        if not item_match:
            continue

        style_code = item_match.group(2)
        metal = "AG925"
        tone = "W"

        # ✅ FIXED: Extract quantity - look for pattern "number space large_number" before dates
        # The pattern is: ... 550 45749.00 01/01/80 17/10/25 ...
        # We want to capture 550 (the quantity before the value)
        qty_value_match = re.search(r'\s(\d{2,4})\s+[\d,]+\.[\d]{2}\s+\d{2}/\d{2}/\d{2}', line)
        order_qty = qty_value_match.group(1) if qty_value_match else ""

        # Initialize fields
        customer_instruction = ""
        special_remarks = ""
        design_instruction = ""
        stamp_instruction = ""
        sku_no = ""
        item_type = ""

        for j in range(i + 1, min(i + 20, len(lines))):
            nxt = lines[j].strip()
            if not nxt:
                continue

            # Customer instruction - extract everything after "Cust.Inst"
            if re.search(r'Cust\.?\s*Inst', nxt, re.IGNORECASE):
                cust_match = re.search(r'Cust\.?\s*Inst\.?\s+(.*?)(?=\s+\d+\s+Plt\s+Rate|$)', nxt, re.IGNORECASE)
                if cust_match:
                    customer_instruction = cust_match.group(1).strip()
                    # Extract item type
                    item_type_match = re.search(r'Fashion\s+(Pendant|Bangle|necklace|ring|earring)s?', customer_instruction, re.IGNORECASE)
                    if item_type_match:
                        item_type = item_type_match.group(1).capitalize()

            # Design/Production Instruction - extract everything after "Prd Inst."
            if re.search(r'Prd\.?\s*Inst', nxt, re.IGNORECASE):
                prd_match = re.search(r'Prd\.?\s*Inst\.?\s+(.*?)(?=\s+SS|$)', nxt, re.IGNORECASE)
                if prd_match:
                    design_instruction = prd_match.group(1).strip()

            # Stamp Instruction - extract everything after "Stmp Inst"
            if re.search(r'Stmp\s*Inst', nxt, re.IGNORECASE):
                stmp_match = re.search(r'Stmp\s*Inst\.?\s+(.*?)(?=\s+Bill\s+of|$)', nxt, re.IGNORECASE)
                if stmp_match:
                    stamp_instruction = stmp_match.group(1).strip()

            # SKU - extract everything after "SKU#"
            if "SKU#" in nxt:
                m = re.search(r'SKU#\s+([\w\s]+?)(?=\s+Prt\s+Cd|$)', nxt)
                if m:
                    sku_no = m.group(1).strip()

            # Special Remarks - extract everything after "Sepcial Rem"
            if re.search(r'Sepcial\s*Rem', nxt, re.IGNORECASE):
                special_rem_match = re.search(r'Sepcial\s*Rem\.?\s+(.*?)(?=\s+Prt\s+Cd|$)', nxt, re.IGNORECASE)
                if special_rem_match:
                    special_remarks = special_rem_match.group(1).strip()

            # Stop if next item line begins
            if re.match(r'^\d+\s+[A-Z0-9]+', nxt):
                break

        # Format SpecialRemarks based on item type
        if item_type:
            special_remarks = f"SKU # {metal}, FASHION {item_type}"

        # ✅ Format StampInstruction based on metal type
        if metal == "AG925":
            stamp_instruction = "925,JD LOGO"
        elif metal == "14KT":
            stamp_instruction = "14,JD LOGO"
        elif metal == "18KT":
            stamp_instruction = "18,JD LOGO"
        else:
            # For any other metal format, try to extract the number
            metal_num = re.search(r'(\d+)', metal)
            if metal_num:
                stamp_instruction = f"{metal_num.group(1)},JD LOGO"
            else:
                stamp_instruction = "JD LOGO"

        # ✅ Set DesignProductionInstruction to "All White" if Tone is "W"
        if tone == "W":
            design_instruction = "All White"

        data.append({
            'SrNo': sr_no,
            'StyleCode': style_code,
            'ItemSize': "",
            'OrderQty': order_qty,
            'OrderItemPcs': "",
            'Metal': metal,
            'Tone': tone,
            'ItemPoNo.': current_item_po_no,
            'ItemRefNo': "",
            'StockType': "",
            'Priority': priority,
            'MakeType': "",
            'CustomerProductionInstruction': customer_instruction,
            'SpecialRemarks': special_remarks,
            'DesignProductionInstruction': design_instruction,
            'StampInstruction': stamp_instruction,
            'OrderGroup': "STERLING JEWELERS(OUTLET)",
            'Certificate': "",
            'SKUNo': sku_no,
            'Basestoneminwt': "",
            'Basestonemaxwt': "",
            'Basemetalminwt': "",
            'Basemetalmaxwt': "",
            'Productiondeliverydate': "",
            'Expecteddeliverydate': "",
            'SetPrice': "",
            'StoneQuality': ""
        })
        sr_no += 1

    return data


# Run and export
processed_data = process_ju_data_with_item_po()
df = pd.DataFrame(processed_data)
if df.empty:
    print("⚠️ No items were extracted. Check regex patterns or input text.")
else:
    print(df[['SrNo','StyleCode','ItemPoNo.','OrderQty','DesignProductionInstruction','SpecialRemarks','SKUNo']])
    output_filename = "JU_Processed_Data_1.xlsx"
    #df.to_excel(output_filename, index=False)
    print(f"\n✅ Data successfully saved to '{output_filename}'")
    print(f"Total items processed: {len(df)}")

   SrNo  StyleCode ItemPoNo. OrderQty DesignProductionInstruction  \
0     1  ABP08249C      4367      550                   All White   
1     2  ABE08249E      4368      550                   All White   
2     3  ABG08249Z      4369      550                   All White   

                 SpecialRemarks SKUNo  
0  SKU # AG925, FASHION Pendant        
1  SKU # AG925, FASHION Earring        
2   SKU # AG925, FASHION Bangle        

✅ Data successfully saved to 'JU_Processed_Data_1.xlsx'
Total items processed: 3


In [4]:
df


,SrNo,StyleCode,ItemSize,OrderQty,OrderItemPcs,Metal,Tone,ItemPoNo.,ItemRefNo,StockType,...,Certificate,SKUNo,Basestoneminwt,Basestonemaxwt,Basemetalminwt,Basemetalmaxwt,Productiondeliverydate,Expecteddeliverydate,SetPrice,StoneQuality
0,1,ABP08249C,,550,,AG925,W,4367,,,...,,,,,,,,,,
1,2,ABE08249E,,550,,AG925,W,4368,,,...,,,,,,,,,,
2,3,ABG08249Z,,550,,AG925,W,4369,,,...,,,,,,,,,,
